In [0]:
# 1. Import dependencies and initialize spark instance
# 2. Load data to df
    # comment to track progress of etl
# 3. Perform data cleansing
    # null values
    # duplicates
    # data type conversion
    # comment when done, error msg if errors
# 4. Check quality metrics
    # min, max values
    # count of records
    # assertions
# 5. Perform data transformation if needed
    # select columns, rename cols
# 6. Save data to delta table

In [0]:
# 1. Import dependencies
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import col, unix_timestamp, round, when, current_timestamp
from datetime import datetime

In [0]:
# 1. Create spark session
spark = SparkSession.builder.appName("db_code_interview_samuli").getOrCreate()

# 2. load data to df
data = [
    ("VTS001", "2024-05-01 08:00:00", "2024-05-01 08:15:00", 3.5, 12.0, "credit_card"),
    ("VTS002", "2024-05-01 09:30:00", "2024-05-01 09:50:00", 5.0, 18.5, "cash"),
    ("VTS003", "2024-05-01 10:00:00", "2024-05-01 10:25:00", 7.0, 24.0, None),
    ("VTS004", None, "2024-05-01 11:15:00", 2.0, 8.0, "credit_card"),
]

schema = StructType([
    StructField("vendor_id", StringType(), True),
    StructField("pickup_datetime", StringType(), True),
    StructField("dropoff_datetime", StringType(), True),
    StructField("trip_distance_miles", DoubleType(), True),
    StructField("fare_amount_usd", DoubleType(), True),
    StructField("payment_type", StringType(), True)
])

df = spark.createDataFrame(data, schema)

In [0]:
# check if critical columns are in the source
required_cols = ["pickup_datetime", "fare_amount_usd"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

In [0]:
# 3. Perform data cleansing
    # null values
    # duplicates
    # data type conversion

df_transformed = (
    df
    .filter(col("pickup_datetime").isNotNull())  # Remove rows with null pickup times
    .filter(col("vendor_id").isNotNull())  # Remove rows with null vendor IDs
    .withColumn("trip_distance_km", round(col("trip_distance_miles") * 1.60934, 2))
    .withColumn("trip_duration_min",
                round((unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60, 1))
    .withColumn("payment_type", when(col("payment_type").isNull(), "unknown").otherwise(col("payment_type")))
    .withColumn("create_date", current_timestamp())
    .withColumn("update_date", current_timestamp())
    .dropDuplicates() # drop duplicates based on all columns
)
# Add comments to track progess
print(f"[INFO] Extracted {df_transformed.count()} rows from source.")


In [0]:
# quality checks    
    # min, max values
    # count of records
df_min_max = df_transformed.selectExpr("min(trip_distance_km) as min_trip_distance_km", "max(trip_distance_km) as max_trip_distance_km", "count(*) as total_records")
df_min_max.display()

# Add assertions to validate data
assert df_transformed.filter(col("fare_amount_usd") < 0).count() == 0, "Negative fares found!"
assert df_transformed.filter(col("fare_amount_usd")) > 10000).count() == 0, "Fares above $10,000 found!"


In [0]:
# simply overwrite
df_transformed.write.format("delta").mode("overwrite").saveAsTable("mycatalog.myschema.target")

In [0]:
# 5. Incremental load to delta table if exist

# Define the target table
target_table = "mycatalog.myschema.target"

# Create a temporary view for df_transformed
df_transformed.createOrReplaceTempView("temp_view")

# Perform the merge operation
spark.sql(f"""
MERGE INTO {target_table} AS target
USING temp_view AS source
ON target.id = source.id  -- Assuming 'id' is the unique identifier for the records
WHEN MATCHED THEN
  UPDATE SET
    target.pickup_datetime = source.pickup_datetime,
    target.vendor_id = source.vendor_id,
    target.trip_distance_km = source.trip_distance_km,
    target.trip_duration_min = source.trip_duration_min,
    target.payment_type = source.payment_type,
    target.update_date = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (
    id,
    pickup_datetime,
    vendor_id,
    trip_distance_km,
    trip_duration_min,
    payment_type,
    update_date
  )
  VALUES (
    source.id,
    source.pickup_datetime,
    source.vendor_id,
    source.trip_distance_km,
    source.trip_duration_min,
    source.payment_type,
    current_timestamp(),
    current_timestamp()
  )
""")

In [0]:
# Incremental load with SCD type 2

from pyspark.sql.functions import col, lit, current_timestamp

# Define the target table
target_table = "mycatalog.myschema.target"

# Create a temporary view for df_transformed
df_transformed.createOrReplaceTempView("temp_view")

# Perform the merge operation for SCD Type 2
spark.sql(f"""
MERGE INTO {target_table} AS target
USING temp_view AS source
ON target.id = source.id AND target.current_flag = 1
WHEN MATCHED AND (
    target.pickup_datetime != source.pickup_datetime OR
    target.vendor_id != source.vendor_id OR
    target.trip_distance_km != source.trip_distance_km OR
    target.trip_duration_min != source.trip_duration_min OR
    target.payment_type != source.payment_type
) THEN
  UPDATE SET
    target.current_flag = 0,
    target.end_date = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (
    id,
    pickup_datetime,
    vendor_id,
    trip_distance_km,
    trip_duration_min,
    payment_type,
    start_date,
    end_date,
    current_flag
  )
  VALUES (
    source.id,
    source.pickup_datetime,
    source.vendor_id,
    source.trip_distance_km,
    source.trip_duration_min,
    source.payment_type,
    current_timestamp(),
    NULL,
    1
  )
""")